# **1. Scaled Dot-Product Attention**

수식 :

$\text{Attention}(Q, K, V) = \text{softmax} \left( \frac{QK^T}{\sqrt{d_k}} \right)V$



In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# Q는 단어 love 임베딩 벡터라고 가정합니다.
Q = torch.tensor([0.1, 0.3, 0.7, 0.9]) # shape: (4,)

K = torch.tensor([
    [0.3, 0.2, 0.8, 0.6], # I
    [0.1, 0.3, 0.7, 0.9], # love
    [0.6, 0.4, 0.5, 0.8], # this
    [0.7, 0.1, 0.4, 0.7], # movie
])
V = K
K

tensor([[0.3000, 0.2000, 0.8000, 0.6000],
        [0.1000, 0.3000, 0.7000, 0.9000],
        [0.6000, 0.4000, 0.5000, 0.8000],
        [0.7000, 0.1000, 0.4000, 0.7000]])

In [7]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_head):
        super().__init__()
        self.d_head = d_head

    def forward(self, Q, K, V, attn_mask=None):
        # ------------------------------------------------------------
        # 1) Attention Score 계산
        # Query와 Key의 관련성을 내적으로 계산합니다.
        # [실습/TODO] Q와 K의 내적을 계산하여 attention_scores를 완성하세요.
        attention_scores = torch.matmul(Q, K.transpose(-2, -1))

        # ------------------------------------------------------------
        # 2) Mask 적용
        # attn_mask가 있다면 mask 값이 0인 위치를 -inf로 바꿉니다.
        # softmax(-inf)는 0이 되므로, 해당 위치는 attention을 받지 않게 됩니다.
        if attn_mask is not None:
            attention_scores = attention_scores.masked_fill(attn_mask == 0, float("-inf"))

        # ------------------------------------------------------------
        # 3) Scaling
        # Attention Score를 sqrt(d_head)로 나눕니다.
        # [실습/TODO] attention_scores를 self.d_head의 제곱근으로 나누어 scaled_scores를 완성하세요.
        scaled_scores = attention_scores / math.sqrt(self.d_head)

        # ------------------------------------------------------------
        # 4) Softmax로 Attention Weight 계산
        # scaled score를 softmax에 넣어 전체 합이 1인 가중치로 변환합니다.
        # attention_weights의 의미:
        # love Query가 각 토큰의 Value를 얼마나 참고할지 나타내는 비율입니다.
        # [실습/TODO] scaled_scores에 softmax 함수를 적용하여 attention_weights를 완성하세요.
        attention_weights = F.softmax(scaled_scores, dim=-1)

        # ------------------------------------------------------------
        # 5) Context Vector 계산
        # Attention Weight를 Value에 곱해 가중합합니다.
        # [실습/TODO] attention_weights와 V를 곱하여 context_vector를 완성하세요.
        context_vector = torch.matmul(attention_weights, V)

        return context_vector, attention_scores, scaled_scores, attention_weights

### 3) 실행



In [3]:
# Q는 현재 shape가 (4,)입니다.
# 행렬곱 결과를 그림처럼 (1, 4)로 보기 위해 (1, 4) 형태로 바꿉니다.
Q_love = Q.reshape(1, 4)

# d_head는 Query/Key 벡터의 차원입니다.
# 여기서는 K의 마지막 차원이 4입니다.
d_head = K.shape[-1]

attention = ScaledDotProductAttention(d_head=d_head)

context_vector, attention_scores, scaled_scores, attention_weights = attention(Q_love, K, V)


# ============================================================
# 4. 결과 확인
# ============================================================

print("Q_love shape:", Q_love.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)
print()

print("1. Attention Scores")
print(attention_scores)
print()

print("2. Scaled Scores")
print(scaled_scores)
print()

print("3. Attention Weights")
print(attention_weights)
print("sum:", attention_weights.sum(dim=-1))
print()

print("4. Context Vector, 즉 Z_love")
print(context_vector)

Q_love shape: torch.Size([1, 4])
K shape: torch.Size([4, 4])
V shape: torch.Size([4, 4])

1. Attention Scores
tensor([[1.1900, 1.4000, 1.2500, 1.0100]])

2. Scaled Scores
tensor([[0.5950, 0.7000, 0.6250, 0.5050]])

3. Attention Weights
tensor([[0.2466, 0.2739, 0.2541, 0.2254]])
sum: tensor([1.])

4. Context Vector, 즉 Z_love
tensor([[0.4116, 0.2557, 0.6062, 0.7555]])


####  출력결과
Q_love shape: torch.Size([1, 4])  
K shape: torch.Size([4, 4])  
V shape: torch.Size([4, 4])  

1. Attention Scores  
tensor([[1.1900, 1.4000, 1.2500, 1.0100]])  

2. Scaled Scores  
tensor([[0.5950, 0.7000, 0.6250, 0.5050]])  

3. Attention Weights  
tensor([[0.2466, 0.2739, 0.2541, 0.2254]])  
sum: tensor([1.])  

4. Context Vector, 즉 Z_love  
tensor([[0.4116, 0.2557, 0.6062, 0.7555]])